# NB03 — Machine Learning Benchmarking: Microbial Source Prediction
**Pipeline:** Generalised Metabolite–Metagenomics Correlation Study  
**Inputs:** `analysis_results.pkl` (from NB02)  
**Outputs:** `ml_results.pkl`, benchmark tables, SHAP figures  
**Cohort:** YACHIDA-CRC-2019 only (stage-stratified CV)  
**Models:** XGBoost · LightGBM · ElasticNet · SVR · Random Forest  

| Step | Analysis |
|---|---|
| 1–2 | Setup, load data |
| 3 | Select top dysregulated metabolite targets |
| 3b | Early vs advanced dysregulation visualisation |
| 4 | Feature matrix: species CLR + clinical confounders (one-hot encoded) |
| 5 | Model definitions (5 models with fallback) |
| 6 | 10-fold stratified CV benchmark — R², Spearman rho, RMSE |
| 7 | Best model selection + benchmark heatmap |
| 8 | SHAP beeswarm (all models, top targets) |
| 9 | SHAP waterfall (per-sample explanation, top targets) |
| 10 | SHAP producer candidate table |
| 10b | Polyamine-focused source attribution |
| 11 | Confounder variance attribution (R² species-only vs full) |
| 12 | Save results |

## 1 · Setup

In [1]:
import sys, warnings, math
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path(".").resolve()))

from utils import (
    INTER_DIR, FIG_DIR, TABLE_DIR,
    DATASET_PRIMARY, FDR_THRESHOLD, N_TOP_TARGETS,
    annotate_pathway,
    load_pickle, save_pickle, savefig,
    PALETTE_STAGE6, PALETTE_3GROUP,
    load_multi_cohort_lodo, DATASETS_LODO, IBD_COHORTS,
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import spearmanr
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.base import clone as sk_clone
from statsmodels.stats.multitest import multipletests

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("SHAP not available — install with: pip install shap")

ana = load_pickle(INTER_DIR / "analysis_results.pkl")

sp_clr = ana["spe"]       # CLR species (YACHIDA, feature-selected)
mt_log = ana["mtb"]       # log10+centred metabolites
meta_y = ana["meta_yk"]
nm_y   = ana["nm_y"]

# Sanitise name map: replace NaN/float values with the KEGG ID string
nm_y = {k: (v if isinstance(v, str) else k) for k, v in nm_y.items()}

for d in [FIG_DIR/"ml", TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Loaded: {sp_clr.shape[1]} species features, {mt_log.shape[1]} metabolite targets")
print(f"Top target count: {N_TOP_TARGETS}")

Loaded: C:\Users\andna\Documents\KI\Results\intermediate\analysis_results.pkl
Loaded: 500 species features, 150 metabolite targets
Top target count: 50


In [2]:
# ── Multi-cohort LODO setup ───────────────────────────────────────────────────
# load_multi_cohort_lodo returns (X, y, groups) across all LODO cohorts in pkl.
# Falls back gracefully if only YACHIDA is present.
pkl_full = load_pickle(INTER_DIR / 'preprocessed_data.pkl')
try:
    X_lodo, y_lodo, lodo_groups = load_multi_cohort_lodo(
        pkl_full, cohorts=None, top_species=300)

    print('LODO cohort composition:')
    print(lodo_groups.value_counts().to_string())
    print(f'\nCommon species features: {X_lodo.shape[1]}')
    print(f'Total samples: {len(X_lodo)}')
    print(f'\nStage.3Group distribution:')
    print(y_lodo.value_counts())

    le_lodo    = LabelEncoder()
    y_lodo_enc = le_lodo.fit_transform(y_lodo.fillna('Healthy'))
    groups_arr = lodo_groups.values
    X_lodo_arr = X_lodo.values.astype(float)
    N_DATASETS = lodo_groups.nunique()

except (ValueError, KeyError) as _e:
    print(f'LODO setup failed: {_e}')
    print('Falling back to within-YACHIDA 10-fold CV.')
    X_lodo_arr = None
    N_DATASETS = 1


Loaded: C:\Users\andna\Documents\KI\Results\intermediate\preprocessed_data.pkl
LODO matrix: 347 samples × 300 species from 1 cohorts
LODO cohort composition:
cohort
YACHIDA-CRC-2019    347

Common species features: 300
Total samples: 347

Stage.3Group distribution:
Stage.3Group
Advanced_CRC    163
Healthy         127
Early_CRC        57
Name: count, dtype: int64


---
## 2 · Top Dysregulated Metabolite Targets
Union of top dysregulated metabolites by |effect_size| from CRC vs Healthy (early + advanced).

In [3]:
da_early = ana["da_mtb"].get("Healthy_vs_Early_CRC",    pd.DataFrame())
da_adv   = ana["da_mtb"].get("Healthy_vs_Advanced_CRC", pd.DataFrame())

def top_n_sig(da_df, n):
    """Return top-n features by |effect_size|, preferring q<0.05."""
    if da_df.empty or "qval" not in da_df.columns:
        return []
    sig = da_df[da_df["qval"] <= FDR_THRESHOLD].copy()
    if len(sig) < n:
        sig = da_df.copy()   # relax if too few pass FDR
    sig["abs_effect"] = sig["effect_size"].abs()
    return sig.nlargest(n, "abs_effect")["feature"].tolist()

early_targets = top_n_sig(da_early, N_TOP_TARGETS)
adv_targets   = top_n_sig(da_adv,   N_TOP_TARGETS)

# Union preserving order (early first)
all_targets = list(dict.fromkeys(early_targets + adv_targets))

# Fallback: use top partial-correlation metabolites
if len(all_targets) < 5 and not ana["corr_partial_sig"].empty:
    from collections import Counter
    counts      = Counter(ana["corr_partial_sig"]["metabolite"])
    all_targets = [m for m, _ in counts.most_common(N_TOP_TARGETS)]

all_targets = [t for t in all_targets if t in mt_log.columns][:N_TOP_TARGETS]

print(f"Targets selected: {len(all_targets)} dysregulated metabolites")
print(f"\nFirst {min(10, len(all_targets))}:")
for t in all_targets[:10]:
    row = da_adv[da_adv["feature"] == t] if not da_adv.empty else pd.DataFrame()
    eff = row["effect_size"].values[0] if not row.empty else float("nan")
    q   = row["qval"].values[0]        if not row.empty else float("nan")
    print(f"  {t}  ({nm_y.get(t,'unknown'):<35})  effect={eff:.3f}  q={q:.3e}")

# Save target list with significance flag
def _is_sig(kegg_id):
    """Check if a target is FDR-significant in either early or advanced DA."""
    for da in [da_early, da_adv]:
        if not da.empty and "qval" in da.columns:
            row = da.loc[da["feature"] == kegg_id, "qval"]
            if not row.empty and (row.values[0] <= FDR_THRESHOLD):
                return True
    return False

target_df = pd.DataFrame({
    "kegg_id":     all_targets,
    "name":        [nm_y.get(t, t) for t in all_targets],
    "pathway":     [annotate_pathway(t) for t in all_targets],
    "significant": [_is_sig(t) for t in all_targets],
})
target_df.to_csv(TABLE_DIR / "ml_target_metabolites.csv", index=False)

n_sig = target_df["significant"].sum()
print(f"\n  FDR-significant: {n_sig}/{len(target_df)}")
if n_sig < len(target_df):
    print(f"  Note: {len(target_df) - n_sig} targets selected by effect-size rank (FDR > 0.05)")

Targets selected: 50 dysregulated metabolites

First 10:
  C07151_Metformin  (C07151                             )  effect=0.122  q=3.101e-01
  C00423_trans-Cinnamate  (C00423                             )  effect=-0.072  q=4.886e-01
  C00122_Fumarate  (C00122                             )  effect=0.160  q=2.097e-01
  C01042_N-Acetylaspartate  (C01042                             )  effect=0.106  q=3.226e-01
  C01959_Taurocyamine  (C01959                             )  effect=0.076  q=4.688e-01
  C08262_Isovalerate  (C08262                             )  effect=0.166  q=2.081e-01
  C00024_Acetyl CoA  (C00024                             )  effect=-0.063  q=5.422e-01
  C00346_Ethanolamine phosphate  (C00346                             )  effect=0.142  q=2.664e-01
  C01879_5-Oxoproline  (C01879                             )  effect=0.013  q=9.078e-01
  C16741_5-Hydroxylysine  (C16741                             )  effect=-0.112  q=3.226e-01

  FDR-significant: 4/50
  Note: 46 targets selec

### Early vs Advanced Dysregulation Profile
Grouped bar chart comparing effect sizes for the selected targets across early-stage and advanced-stage CRC vs Healthy.

In [4]:
# Merge early and advanced effect sizes for selected targets
viz_df = pd.DataFrame({"target": all_targets})
if not da_early.empty:
    viz_df = viz_df.merge(
        da_early[["feature", "effect_size", "qval"]].rename(
            columns={"feature": "target", "effect_size": "early_effect", "qval": "early_q"}),
        on="target", how="left")
else:
    viz_df["early_effect"] = float("nan")
    viz_df["early_q"] = float("nan")

if not da_adv.empty:
    viz_df = viz_df.merge(
        da_adv[["feature", "effect_size", "qval"]].rename(
            columns={"feature": "target", "effect_size": "adv_effect", "qval": "adv_q"}),
        on="target", how="left")
else:
    viz_df["adv_effect"] = float("nan")
    viz_df["adv_q"] = float("nan")

viz_df["name"] = viz_df["target"].map(nm_y)
viz_df = viz_df.sort_values("adv_effect", ascending=True, na_position="first")

# Grouped horizontal bar chart
fig, ax = plt.subplots(figsize=(10, max(5, len(viz_df) * 0.4)))
y_pos = np.arange(len(viz_df))
bar_h = 0.35
ax.barh(y_pos - bar_h / 2, viz_df["early_effect"].fillna(0), bar_h,
        label="Early CRC vs Healthy", color="#66BB6A", edgecolor="white", lw=0.5)
ax.barh(y_pos + bar_h / 2, viz_df["adv_effect"].fillna(0), bar_h,
        label="Advanced CRC vs Healthy", color="#EF5350", edgecolor="white", lw=0.5)

# Mark FDR-significant bars with asterisk
for i, (_, row) in enumerate(viz_df.iterrows()):
    if pd.notna(row.get("early_q")) and row["early_q"] <= FDR_THRESHOLD:
        ax.text(row["early_effect"] + 0.005, i - bar_h / 2, "*", va="center", fontsize=10)
    if pd.notna(row.get("adv_q")) and row["adv_q"] <= FDR_THRESHOLD:
        ax.text(row["adv_effect"] + 0.005, i + bar_h / 2, "*", va="center", fontsize=10)

ax.set_yticks(y_pos)
ax.set_yticklabels(viz_df["name"].str[:40], fontsize=8)
ax.set_xlabel("Effect size (rank-biserial r)")
ax.set_title("Dysregulated Metabolites: Early vs Advanced CRC", fontweight="bold")
ax.legend(loc="lower right")
ax.axvline(0, color="black", lw=0.5)
plt.tight_layout()
savefig(fig, "ml", "nb03_early_vs_advanced_dysregulation.png")
print("Saved: nb03_early_vs_advanced_dysregulation.png")

Saved figure: C:\Users\andna\Documents\KI\Results\figures\ml\nb03_early_vs_advanced_dysregulation.png
Saved: nb03_early_vs_advanced_dysregulation.png


---
## 3 · Feature Matrix & Confounders

In [5]:
conf_cols  = ["Age", "BMI", "Gender", "Alcohol", "Tumor location"]
avail_conf = [c for c in conf_cols if c in meta_y.columns]

X_species = sp_clr.copy()
if avail_conf:
    cov = meta_y[avail_conf].copy()
    # One-hot encode nominal features (avoids false ordinal from LabelEncoder)
    cat_cols = cov.select_dtypes("object").columns.tolist()
    if cat_cols:
        cov = pd.get_dummies(cov, columns=cat_cols, drop_first=True, dtype=float)
    # Leave NaN in place — imputation happens per-fold inside CV (no data leakage)
    X_full = pd.concat([X_species, cov], axis=1)
    conf_col_indices = [X_full.columns.get_loc(c) for c in cov.columns]
else:
    X_full = X_species.copy()
    conf_col_indices = []

# Stage labels for stratified CV — exclude NaN and non-3-group labels
valid_mask  = (
    meta_y["Stage.3Group"].notna() &
    meta_y["Stage.3Group"].isin(["Healthy", "Early_CRC", "Advanced_CRC"])
)
X_arr_cv   = X_full.values[valid_mask]
stage_enc  = LabelEncoder().fit_transform(meta_y["Stage.3Group"][valid_mask])

print(f"Feature matrix      : {X_full.shape}")
print(f"CV sample size      : {valid_mask.sum()} / {len(valid_mask)} (NaN stage excluded)")
print(f"Confounders included: {avail_conf}")
if conf_col_indices:
    print(f"Confounder columns  : {len(conf_col_indices)} (one-hot encoded)")

Feature matrix      : (347, 510)
CV sample size      : 347 / 347 (NaN stage excluded)
Confounders included: ['Age', 'BMI', 'Gender', 'Alcohol', 'Tumor location']
Confounder columns  : 10 (one-hot encoded)


---
## 4 · Model Definitions

In [6]:
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline

def get_xgboost():
    try:
        from xgboost import XGBRegressor
        return XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05,
                            random_state=42, verbosity=0, n_jobs=4)
    except ImportError:
        print("xgboost not installed; substituting ExtraTreesRegressor")
        return ExtraTreesRegressor(n_estimators=200, random_state=42, n_jobs=4)

def get_lgbm():
    try:
        from lightgbm import LGBMRegressor
        return LGBMRegressor(n_estimators=200, learning_rate=0.05,
                             num_leaves=31, random_state=42, verbosity=-1, n_jobs=4)
    except ImportError:
        print("lightgbm not installed; substituting GradientBoostingRegressor")
        return GradientBoostingRegressor(
            n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)

models = {
    "XGBoost":    get_xgboost(),
    "LightGBM":   get_lgbm(),
    "ElasticNet": Pipeline([("scaler", StandardScaler()),
                             ("en", ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=5000))]),
    "SVR":        Pipeline([("scaler", StandardScaler()),
                             ("svr", SVR(kernel="rbf", C=1.0, epsilon=0.1))]),
    "RF":         RandomForestRegressor(n_estimators=200, max_depth=None,
                                         random_state=42, n_jobs=4),
}
print(f"Models registered: {list(models.keys())}")


Models registered: ['XGBoost', 'LightGBM', 'ElasticNet', 'SVR', 'RF']


---
## 5 · 10-Fold Stratified Cross-Validation Benchmark

In [7]:
# ── Choose CV strategy: LODO if >=2 datasets loaded, else 10-fold within-YACHIDA ─
if X_lodo_arr is not None and N_DATASETS >= 2:
    print(f'Running LODO cross-validation across {N_DATASETS} datasets')
    cv_strategy   = 'LODO'
    cv_iter       = list(LeaveOneGroupOut().split(X_lodo_arr, y_lodo_enc, groups=groups_arr))
    X_cv          = X_lodo_arr
    feature_names = list(X_lodo.columns)
else:
    print('Only YACHIDA available — using 10-fold stratified CV')
    cv_strategy   = '10-fold'
    cv_iter       = list(StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
                         .split(X_arr_cv, stage_enc))
    X_cv          = X_arr_cv
    feature_names = (
        list(sp_clr.columns)
        + (list(pd.get_dummies(meta_y[avail_conf], drop_first=True).columns)
           if avail_conf else [])
    )

benchmark_rows: list = []
oof_shap_store: dict = {}

def _impute_fold(X_train, X_test, col_indices):
    """Impute NaN in confounder columns using training-fold medians only."""
    if not col_indices:
        return X_train, X_test
    X_tr, X_te = X_train.copy(), X_test.copy()
    train_medians = np.nanmedian(X_tr[:, col_indices], axis=0)
    for j, ci in enumerate(col_indices):
        X_tr[np.isnan(X_tr[:, ci]), ci] = train_medians[j]
        X_te[np.isnan(X_te[:, ci]), ci] = train_medians[j]
    return X_tr, X_te

for target in all_targets:
    y = mt_log[target].values[valid_mask] if cv_strategy == '10-fold' else mt_log[target].values
    target_name = nm_y.get(target, target)

    for model_name, model in models.items():
        r2_scores, rho_scores, rmse_scores = [], [], []
        oof_shap_vals: list = []
        oof_test_indices: list = []

        for train_idx, test_idx in cv_iter:
            if cv_strategy == '10-fold':
                X_tr, X_te = _impute_fold(X_cv[train_idx], X_cv[test_idx], conf_col_indices)
            else:
                X_tr, X_te = X_cv[train_idx], X_cv[test_idx]

            m      = sk_clone(model)
            m.fit(X_tr, y[train_idx])
            y_pred = m.predict(X_te)

            r2_scores.append(r2_score(y[test_idx], y_pred))
            rho, _ = spearmanr(y[test_idx], y_pred)
            rho_scores.append(rho if not math.isnan(rho) else 0.0)
            rmse_scores.append(math.sqrt(mean_squared_error(y[test_idx], y_pred)))

            # OOF SHAP — tree models only (SVR/ElasticNet too slow for LODO)
            if SHAP_AVAILABLE and model_name in ('RF', 'XGBoost', 'LightGBM'):
                try:
                    _base = (m.named_steps.get(model_name.lower(), m)
                             if hasattr(m, 'named_steps') else m)
                    exp   = shap.TreeExplainer(_base)
                    sv    = exp.shap_values(X_te)
                    oof_shap_vals.append(sv)
                    oof_test_indices.extend(test_idx.tolist())
                except Exception:
                    pass

        if oof_shap_vals:
            all_sv = np.vstack(oof_shap_vals)
            oof_shap_store[(target, model_name)] = {
                'indices':       oof_test_indices,
                'shap_matrix':   all_sv,
                'mean_abs_shap': np.abs(all_sv).mean(axis=0),
            }

        benchmark_rows.append({
            'target':      target,
            'target_name': target_name,
            'model':       model_name,
            'cv_strategy': cv_strategy,
            'r2_mean':     np.mean(r2_scores),
            'r2_std':      np.std(r2_scores),
            'rho_mean':    np.mean(rho_scores),
            'rho_std':     np.std(rho_scores),
            'rmse_mean':   np.mean(rmse_scores),
            'rmse_std':    np.std(rmse_scores),
            'pathway':     annotate_pathway(target),
        })
    print(f'  {target_name[:50]}')

benchmark_df = pd.DataFrame(benchmark_rows)
benchmark_df.to_csv(TABLE_DIR / 'ml_benchmark.csv', index=False)
print(f'\nBenchmark complete: {len(benchmark_df)} model x target combinations ({cv_strategy} CV)')


Only YACHIDA available — using 10-fold stratified CV
  C07151
  C00423
  C00122
  C01042
  C01959
  C08262
  C00024
  C00346
  C01879
  C16741
  C08277
  C01015
  C02037
  _DCA
  C00879
  C01551
  C00630
  C02378
  C00086
  C01425
  C00491
  C05332
  C01118
  C00245
  C02714
  C02155
  C09815
  C00191
  3-Indoxyl sulfate
  C01104
  C02230
  C00334
  C00311
  C02679
  C00003
  C00092
  C00123
  C02710
  C05984
  C03264
  C00407
  C00383
  C00186
  C00449
  C00255
  C00785
  C11118
  C17714
  C00417
  C00314

Benchmark complete: 250 model x target combinations (10-fold CV)


In [8]:
# ── OOF SHAP summary (cross-validated feature importances) ──────────────────
if oof_shap_store:
    shap_oof_rows = []
    for (target, model_name), store in oof_shap_store.items():
        top_idx = np.argsort(store['mean_abs_shap'])[::-1][:10]
        for rank, fi in enumerate(top_idx):
            shap_oof_rows.append({
                'target':            target,
                'model':             model_name,
                'cv_strategy':       cv_strategy,
                'rank':              rank + 1,
                'feature':           feature_names[fi] if fi < len(feature_names) else f'feat_{fi}',
                'mean_abs_shap_oof': float(store['mean_abs_shap'][fi]),
            })
    oof_shap_df = pd.DataFrame(shap_oof_rows)
    oof_shap_df.to_csv(TABLE_DIR / 'lodo_oof_shap_summary.csv', index=False)
    print(f'OOF SHAP summary: {len(oof_shap_df)} rows saved to lodo_oof_shap_summary.csv')
    print(oof_shap_df.head(20).to_string(index=False))
else:
    print('No OOF SHAP values collected (non-tree models only, or SHAP unavailable).')


OOF SHAP summary: 1500 rows saved to lodo_oof_shap_summary.csv
          target    model cv_strategy  rank                        feature  mean_abs_shap_oof
C07151_Metformin  XGBoost     10-fold     1           Prevotella stercorea           0.028759
C07151_Metformin  XGBoost     10-fold     2          Veillonella parvula_A           0.026212
C07151_Metformin  XGBoost     10-fold     3            UMGS687 sp900545735           0.025115
C07151_Metformin  XGBoost     10-fold     4    Catenibacterium sp900764565           0.020797
C07151_Metformin  XGBoost     10-fold     5          Lachnospira rogosae_A           0.020345
C07151_Metformin  XGBoost     10-fold     6         CCUG-7971 spG000499525           0.019776
C07151_Metformin  XGBoost     10-fold     7 Faecalibacterium prausnitzii_G           0.017899
C07151_Metformin  XGBoost     10-fold     8          Mitsuokella multacida           0.017861
C07151_Metformin  XGBoost     10-fold     9           Fimisoma sp900754795           0.0162

---
## 6 · Best Model Selection & Benchmark Heatmap

In [9]:
best_per_target = (
    benchmark_df
    .sort_values("rho_mean", ascending=False)
    .groupby("target")
    .first()
    .reset_index()[["target", "target_name", "model", "rho_mean", "r2_mean", "pathway"]]
)
best_per_target.to_csv(TABLE_DIR / "ml_best_model_per_target.csv", index=False)

# Benchmark heatmap — rho by model x metabolite
pivot = benchmark_df.pivot_table(
    index="target_name", columns="model", values="rho_mean")
fig, ax = plt.subplots(figsize=(11, max(7, len(pivot) * 0.32)))
sns.heatmap(pivot, annot=True, fmt=".2f", cmap="RdYlGn", center=0,
            linewidths=0.4, ax=ax, cbar_kws={"label": "Mean Spearman rho (10-fold CV)"},
            vmin=-0.2, vmax=0.8)
ax.set_title("ML Benchmark — All Models x Top Dysregulated Metabolites", fontweight="bold")
ax.set_ylabel(""); ax.set_xlabel("")
plt.tight_layout()
savefig(fig, "ml", "nb03_benchmark_heatmap.png")

print("Best models per target:")
print(best_per_target[["target_name","model","rho_mean","r2_mean"]].to_string(index=False))
print("\nBest model distribution:")
print(best_per_target["model"].value_counts().to_string())


Saved figure: C:\Users\andna\Documents\KI\Results\figures\ml\nb03_benchmark_heatmap.png
Best models per target:
      target_name      model  rho_mean   r2_mean
3-Indoxyl sulfate   LightGBM  0.443276  0.122067
           C00003    XGBoost  0.394804  0.057533
           C00024        SVR  0.344420  0.068959
           C00086        SVR  0.328689  0.060935
           C00092   LightGBM  0.115095 -0.109711
           C00122        SVR  0.336934  0.086195
           C00123        SVR  0.478100  0.200225
           C00186        SVR  0.307311  0.020498
           C00191   LightGBM  0.279461  0.120071
           C00245         RF  0.450308  0.131799
           C00255         RF  0.202513  0.003044
           C00311        SVR  0.224619 -0.042406
           C00314         RF  0.407003  0.140390
           C00334        SVR  0.496593  0.243003
           C00346         RF  0.201335 -0.127276
           C00383         RF  0.364542  0.067978
           C00407        SVR  0.439246  0.190637
      

## Performance Dashboard — All 5 Models

Comprehensive comparison of XGBoost, LightGBM, ElasticNet, SVR, and Random Forest across 50 dysregulated metabolite targets using 10-fold stratified cross-validation. Metrics: R² (coefficient of determination), Spearman ρ (rank correlation), and RMSE (root mean squared error).

In [ ]:
# ── Model Performance Dashboard ────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

bench = pd.read_csv(TABLE_DIR / "ml_benchmark.csv")

MODEL_ORDER  = ["XGBoost", "LightGBM", "ElasticNet", "SVR", "RF"]
PALETTE      = sns.color_palette("Set2", 5)
MODEL_COLORS = dict(zip(MODEL_ORDER, PALETTE))

sns.set_theme(style="whitegrid", context="paper", font_scale=1.1)

# ── Figure 1: Violin plots — metric distribution per model ─────────────────
metrics = [
    ("r2_mean",   "R² (Coefficient of Determination)"),
    ("rho_mean",  "Spearman ρ"),
    ("rmse_mean", "RMSE"),
]
fig1, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, (col, label) in zip(axes, metrics):
    sns.violinplot(data=bench, x="model", y=col, order=MODEL_ORDER,
                   palette=PALETTE, inner="box", cut=0, ax=ax)
    ax.axhline(0, color="black", lw=0.8, ls="--", alpha=0.5)
    ax.set_xlabel("Model"); ax.set_ylabel(label); ax.set_title(label)
    ax.tick_params(axis="x", rotation=30)
fig1.suptitle("5-Model Performance Comparison (10-fold CV, 50 targets)", y=1.02)
fig1.tight_layout()
fig1.savefig(FIG_DIR / "ml" / "nb03_model_comparison_violin.png", dpi=150, bbox_inches="tight")
plt.show(); print("Saved: nb03_model_comparison_violin.png")

# ── Figure 2: Win-rate bar chart ───────────────────────────────────────────
best_model = (bench.loc[bench.groupby("target")["rho_mean"].idxmax(), "model"]
                   .value_counts()
                   .reindex(MODEL_ORDER, fill_value=0))
fig2, ax2 = plt.subplots(figsize=(6, 4))
bars = ax2.bar(MODEL_ORDER, best_model.values, color=PALETTE, edgecolor="white", width=0.6)
for bar, v in zip(bars, best_model.values):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
             str(v), ha="center", va="bottom", fontsize=10)
ax2.set_xlabel("Model")
ax2.set_ylabel("Targets where model is best (Spearman ρ)")
ax2.set_title("Win Rate: Best Model per Metabolite Target")
fig2.tight_layout()
fig2.savefig(FIG_DIR / "ml" / "nb03_model_win_rate.png", dpi=150, bbox_inches="tight")
plt.show(); print("Saved: nb03_model_win_rate.png")

# ── Figure 3: Radar / spider chart ────────────────────────────────────────
agg = bench.groupby("model").agg(
    mean_r2     =("r2_mean",   lambda x: x.clip(0).mean()),
    mean_rho    =("rho_mean",  "mean"),
    mean_rmse   =("rmse_mean", "mean"),
    frac_pos_r2 =("r2_mean",   lambda x: (x > 0).mean()),
).reindex(MODEL_ORDER)
agg["win_rate"]   = best_model / best_model.max()
rmse_range        = agg["mean_rmse"].max() - agg["mean_rmse"].min() + 1e-9
agg["rmse_score"] = 1 - (agg["mean_rmse"] - agg["mean_rmse"].min()) / rmse_range

radar_cols   = ["mean_r2", "mean_rho", "rmse_score", "frac_pos_r2", "win_rate"]
radar_labels = ["Mean R²", "Mean Spearman ρ", "RMSE\n(inverted)", "Frac R²>0", "Win rate"]
N      = len(radar_cols)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist() + [0]

fig3, ax3 = plt.subplots(figsize=(7, 7), subplot_kw={"polar": True})
for model, color in MODEL_COLORS.items():
    vals = agg.loc[model, radar_cols].clip(0, 1).tolist()
    vals += vals[:1]
    ax3.plot(angles, vals, "-o", label=model, color=color, lw=2)
    ax3.fill(angles, vals, alpha=0.07, color=color)
ax3.set_xticks(angles[:-1])
ax3.set_xticklabels(radar_labels, size=9)
ax3.set_ylim(0, 1)
ax3.set_title("Model Performance Radar (all metrics normalised 0–1)", pad=20)
ax3.legend(loc="upper right", bbox_to_anchor=(1.4, 1.15))
fig3.tight_layout()
fig3.savefig(FIG_DIR / "ml" / "nb03_model_radar.png", dpi=150, bbox_inches="tight")
plt.show(); print("Saved: nb03_model_radar.png")

# ── Figure 4: Full Spearman ρ heatmap ─────────────────────────────────────
pivot = bench.pivot_table(index="target_name", columns="model",
                          values="rho_mean")[MODEL_ORDER]
pivot = pivot.sort_values("SVR", ascending=False)
fig4, ax4 = plt.subplots(figsize=(8, 18))
sns.heatmap(pivot, cmap="RdYlGn", center=0, vmin=-0.3, vmax=0.7,
            linewidths=0.3, linecolor="white",
            cbar_kws={"label": "Spearman ρ (10-fold CV)"}, ax=ax4)
ax4.set_title("Spearman ρ — All 5 Models × 50 Dysregulated Metabolites")
ax4.set_xlabel("Model"); ax4.set_ylabel("Metabolite")
ax4.tick_params(axis="y", labelsize=7)
fig4.tight_layout()
fig4.savefig(FIG_DIR / "ml" / "nb03_model_full_heatmap.png", dpi=150, bbox_inches="tight")
plt.show(); print("Saved: nb03_model_full_heatmap.png")

# ── Summary statistics table ───────────────────────────────────────────────
summary = bench.groupby("model").agg(
    Mean_R2     =("r2_mean",   "mean"),
    Median_R2   =("r2_mean",   "median"),
    Frac_R2_pos =("r2_mean",   lambda x: (x > 0).mean()),
    Mean_rho    =("rho_mean",  "mean"),
    Median_rho  =("rho_mean",  "median"),
    Mean_RMSE   =("rmse_mean", "mean"),
).reindex(MODEL_ORDER).round(3)
print(summary.to_string())
summary.to_csv(TABLE_DIR / "ml_model_performance_summary.csv")
print("\nSaved: ml_model_performance_summary.csv")


---
## 7 · SHAP — Fit Best Models on Full CV Sample Set

In [10]:
# Impute full X_arr_cv for refit (using all-sample medians — no CV leakage concern here)
X_arr_refit = X_arr_cv.copy()
if conf_col_indices:
    all_medians = np.nanmedian(X_arr_refit[:, conf_col_indices], axis=0)
    for j, ci in enumerate(conf_col_indices):
        mask = np.isnan(X_arr_refit[:, ci])
        X_arr_refit[mask, ci] = all_medians[j]

# Refit best model per target on the full valid-mask sample set
fitted_models = {}
feature_names  = list(X_full.columns)

for _, row in best_per_target.iterrows():
    target     = row["target"]
    model_name = row["model"]
    y          = mt_log[target].values[valid_mask]
    best_m     = sk_clone(models[model_name])
    best_m.fit(X_arr_refit, y)
    fitted_models[target] = {"model": best_m, "model_name": model_name, "y": y}

print(f"Fitted {len(fitted_models)} models.")

Fitted 50 models.


---
## 8 · SHAP Beeswarm Plots (all models, top targets)
Shows distribution of SHAP values across samples — which species consistently push predictions up or down.

In [11]:
shap_results = {}

if SHAP_AVAILABLE:
    # Subsample for SVR KernelExplainer (documented approximation)
    N_KERNEL_BG   = 50
    N_KERNEL_TEST = min(100, X_arr_refit.shape[0])

    for target, fit_info in fitted_models.items():
        best_m      = fit_info["model"]
        model_name  = fit_info["model_name"]
        target_name = nm_y.get(target, target)

        try:
            if model_name in ["XGBoost", "LightGBM", "RF"]:
                explainer = shap.TreeExplainer(best_m)
                shap_vals = explainer.shap_values(X_arr_refit)
                try:
                    exp_obj = explainer(X_arr_refit)
                except (TypeError, AttributeError):
                    exp_obj = shap.Explanation(
                        values=shap_vals,
                        base_values=explainer.expected_value,
                        data=X_arr_refit,
                        feature_names=feature_names,
                    )

            elif model_name == "ElasticNet":
                X_scaled  = best_m.named_steps["scaler"].transform(X_arr_refit)
                explainer = shap.LinearExplainer(
                    best_m.named_steps["en"], X_scaled,
                    feature_perturbation="observational")
                shap_vals = explainer.shap_values(X_scaled)
                try:
                    exp_obj = explainer(X_scaled)
                except (TypeError, AttributeError):
                    exp_obj = shap.Explanation(
                        values=shap_vals,
                        base_values=explainer.expected_value,
                        data=X_scaled,
                        feature_names=feature_names,
                    )

            elif model_name == "SVR":
                background = shap.sample(X_arr_refit, N_KERNEL_BG)
                explainer  = shap.KernelExplainer(best_m.predict, background)
                shap_vals  = explainer.shap_values(X_arr_refit[:N_KERNEL_TEST])
                exp_obj    = None   # KernelExplainer does not return Explanation objects

            else:
                continue

            mean_abs = np.abs(shap_vals).mean(axis=0)
            top_idx  = np.argsort(mean_abs)[::-1][:20]
            top_feat = [(feature_names[i], float(mean_abs[i])) for i in top_idx]

            shap_results[target] = {
                "model":        model_name,
                "shap_vals":    shap_vals,
                "exp_obj":      exp_obj,
                "top_features": top_feat,
                "X_used":       X_arr_refit[:N_KERNEL_TEST] if model_name == "SVR" else X_arr_refit,
            }

            # ---- Beeswarm plot ----
            X_disp  = X_arr_refit[:N_KERNEL_TEST] if model_name == "SVR" else X_arr_refit
            sv_disp = shap_vals

            top_feat_names = [f[0] for f in top_feat[:15]]
            top_idx_plot, labels = [], []
            for fn in top_feat_names:
                if fn in feature_names:
                    top_idx_plot.append(feature_names.index(fn))
                    labels.append(fn[:35])

            sv_top = sv_disp[:, top_idx_plot]
            X_top  = X_disp[:, top_idx_plot]

            # Let shap.summary_plot create its own figure (no orphan fig/ax)
            shap.summary_plot(
                sv_top, X_top, feature_names=labels,
                show=False, plot_type="dot", max_display=15
            )
            fig = plt.gcf()
            fig.set_size_inches(9, max(5, len(labels) * 0.35))
            plt.title(
                f"SHAP Beeswarm — {target_name[:40]}\n({model_name})",
                fontweight="bold")
            plt.tight_layout()
            safe_name = target.replace("/", "-")
            plt.savefig(
                FIG_DIR / "ml" / f"nb03_shap_beeswarm_{safe_name}_{model_name}.png",
                dpi=120, bbox_inches="tight")
            plt.close("all")
            print(f"  SHAP beeswarm: {target_name[:40]} ({model_name})")

        except Exception as e:
            print(f"  SHAP failed for {target_name}: {e}")

    print(f"\nSHAP computed for {len(shap_results)}/{len(fitted_models)} targets")
else:
    print("Skipping SHAP — install: pip install shap")

  SHAP beeswarm: 3-Indoxyl sulfate (LightGBM)
  SHAP beeswarm: C00003 (XGBoost)


  0%|          | 0/100 [00:00<?, ?it/s]

  SHAP beeswarm: C00024 (SVR)


  0%|          | 0/100 [00:00<?, ?it/s]

  SHAP beeswarm: C00086 (SVR)
  SHAP beeswarm: C00092 (LightGBM)


...

  SHAP beeswarm: C16741 (SVR)


  0%|          | 0/100 [00:00<?, ?it/s]

  SHAP beeswarm: C17714 (SVR)


  0%|          | 0/100 [00:00<?, ?it/s]

  SHAP beeswarm: _DCA (SVR)

SHAP computed for 49/50 targets


---
## 9 · SHAP Waterfall Plots (per-sample explanation, top targets)
Waterfall shows how each feature pushes a single sample's prediction from the baseline (expected value) to the final output. One plot per top target, showing the sample closest to the group median.

In [12]:
if SHAP_AVAILABLE:
    if not shap_results:
        print("No SHAP results — waterfall plots skipped.")
    else:
        # Produce waterfall for top 10 targets (by best rho)
        top_waterfall_targets = best_per_target.nlargest(
            min(10, len(best_per_target)), "rho_mean")["target"].tolist()

        for target in top_waterfall_targets:
            if target not in shap_results:
                continue
            res         = shap_results[target]
            model_name  = res["model"]
            exp_obj     = res["exp_obj"]
            target_name = nm_y.get(target, target)

            # Guard: SVR and any failed Explanation object
            if exp_obj is None:
                print(f"  Waterfall skipped: {target_name} ({model_name} — no Explanation object)")
                continue

            try:
                y_pred_all  = fitted_models[target]["model"].predict(X_arr_refit)
                median_pred = np.median(y_pred_all)
                sample_idx  = int(np.argmin(np.abs(y_pred_all - median_pred)))

                # Let shap.waterfall_plot create its own figure (no orphan fig/ax)
                shap.waterfall_plot(exp_obj[sample_idx], max_display=15, show=False)
                fig = plt.gcf()
                fig.set_size_inches(9, 6)
                plt.title(
                    f"SHAP Waterfall — {target_name[:40]}\n"
                    f"({model_name}, sample near median prediction)",
                    fontweight="bold")
                plt.tight_layout()
                safe_name = target.replace("/", "-")
                plt.savefig(
                    FIG_DIR / "ml" / f"nb03_shap_waterfall_{safe_name}_{model_name}.png",
                    dpi=120, bbox_inches="tight")
                plt.close("all")
                print(f"  Waterfall saved: {target_name[:40]} ({model_name})")

            except Exception as e:
                print(f"  Waterfall failed: {target_name} — {e}")
else:
    print("Skipping waterfall — SHAP not available")

  Waterfall skipped: C02714 (SVR — no Explanation object)
  Waterfall saved: C08277 (RF)
  Waterfall skipped: C01042 (SVR — no Explanation object)
  Waterfall skipped: C05332 (SVR — no Explanation object)
  Waterfall skipped: C00334 (SVR — no Explanation object)
  Waterfall skipped: C17714 (SVR — no Explanation object)
  Waterfall skipped: C00123 (SVR — no Explanation object)
  Waterfall saved: C00245 (RF)
  Waterfall saved: 3-Indoxyl sulfate (LightGBM)
  Waterfall skipped: C00407 (SVR — no Explanation object)


---
## 10 · SHAP Producer Candidate Table

In [13]:
shap_long_rows = []
for target, res in shap_results.items():
    for feat, importance in res["top_features"]:
        if feat in sp_clr.columns:   # species features only (not confounders)
            shap_long_rows.append({
                "target":          target,
                "target_name":     nm_y.get(target, target),
                "model":           res["model"],
                "species":         feat,
                "shap_importance": importance,
                "pathway":         annotate_pathway(target),
            })

shap_long = pd.DataFrame(shap_long_rows)

# Wide-format SHAP summary: rows = metabolites, cols = species
# (required by NB05 melt and NB06 species ranking)
if not shap_long.empty:
    shap_summary = shap_long.pivot_table(
        index="target", columns="species", values="shap_importance",
        aggfunc="mean"
    ).fillna(0.0)
    # Reset column axis name so stack() gives clean column names
    shap_summary.columns.name = None
    shap_summary.index.name   = "metabolite"

    shap_long = shap_long.sort_values("shap_importance", ascending=False)
    shap_long.to_csv(TABLE_DIR / "shap_producer_candidates.csv", index=False)

    print(f"SHAP producer candidates: {len(shap_long)} entries")
    print(f"SHAP wide summary shape : {shap_summary.shape}  (metabolites × species)")
    print(f"Unique species : {shap_long['species'].nunique()}")
    print(f"Unique targets : {shap_long['target'].nunique()}")
    print("\nTop 15 (by SHAP importance):")
    print(shap_long[["species","target_name","model","shap_importance"]].head(15).to_string(index=False))
else:
    shap_summary = pd.DataFrame()
    print("No SHAP results — check above cells.")

SHAP producer candidates: 960 entries
SHAP wide summary shape : (49, 372)  (metabolites × species)
Unique species : 372
Unique targets : 49

Top 15 (by SHAP importance):
                       species       target_name    model  shap_importance
           UBA5905 sp900763035            C08277       RF         0.236887
        Aphodousia sp900553105            C00423       RF         0.159896
           UMGS687 sp900545735            C01118       RF         0.146906
         Bilophila wadsworthia            C00245       RF         0.110351
           UMGS693 sp900544555            C02378       RF         0.103473
     Acutalibacter sp910580045            C11118       RF         0.099357
    Scatavimonas merdipullorum            C00314       RF         0.084577
           UMGS687 sp900545735            C00383       RF         0.080201
           CAG-170 sp000432135            C00449       RF         0.077694
        Aphodousia sp900553105            C00383       RF         0.064008
Erysi

---
## 10b · Polyamine-Focused Source Attribution
Flag any selected targets belonging to the polyamine pathway (putrescine, spermidine, spermine, agmatine, cadaverine, ornithine, arginine) and show their top microbial producers by SHAP importance.

In [14]:
from utils import pathway_kegg_ids

polyamine_ids = set(pathway_kegg_ids("Polyamine").keys())
poly_targets  = [t for t in all_targets if t in polyamine_ids]

if poly_targets:
    print(f"Polyamine targets among selected: {len(poly_targets)}")
    for t in poly_targets:
        print(f"  {t} — {nm_y.get(t, t)} ({annotate_pathway(t)})")

    # Subset SHAP results to polyamines
    poly_shap = shap_long[shap_long["target"].isin(poly_targets)].copy()
    if not poly_shap.empty:
        n_poly = len(poly_targets)
        fig, axes = plt.subplots(1, n_poly,
                                 figsize=(6 * n_poly, max(5, 10 * 0.35)),
                                 squeeze=False)
        for i, t in enumerate(poly_targets):
            sub = poly_shap[poly_shap["target"] == t].nlargest(10, "shap_importance")
            ax = axes[0, i]
            y_pos = range(len(sub))
            ax.barh(list(y_pos), sub["shap_importance"].values, color="#7E57C2")
            ax.set_yticks(list(y_pos))
            ax.set_yticklabels(sub["species"].str[:30].values, fontsize=8)
            ax.set_xlabel("Mean |SHAP|")
            ax.set_title(f"{nm_y.get(t, t)[:30]}\n({sub['model'].iloc[0]})",
                         fontweight="bold", fontsize=10)
            ax.invert_yaxis()
        plt.suptitle("Polyamine Microbial Source Attribution (SHAP)",
                     fontweight="bold", y=1.02)
        plt.tight_layout()
        savefig(fig, "ml", "nb03_polyamine_shap_attribution.png")
        print("Saved: nb03_polyamine_shap_attribution.png")

        # Summary table
        poly_summary = poly_shap.groupby(["target", "target_name", "species"]).agg(
            shap_importance=("shap_importance", "mean"),
            model=("model", "first"),
        ).reset_index().sort_values(["target", "shap_importance"], ascending=[True, False])
        poly_summary.to_csv(TABLE_DIR / "polyamine_shap_producers.csv", index=False)
        print(f"\nPolyamine producer table saved: {len(poly_summary)} rows")
    else:
        print("No SHAP results available for polyamine targets.")
else:
    print("No polyamine metabolites among the selected targets.")

No polyamine metabolites among the selected targets.


---
## 11 · Confounder Variance Attribution (species-only vs full model)

In [15]:
X_species_cv = sp_clr.values[valid_mask]
var_attr_rows = []

# Species-only feature matrix has no NaN (CLR values), but full matrix needs fold-wise imputation
for target in all_targets[:min(15, len(all_targets))]:
    y          = mt_log[target].values[valid_mask]
    target_name = nm_y.get(target, target)
    tgt_row    = best_per_target[best_per_target["target"] == target]
    best_mod   = tgt_row["model"].values[0] if len(tgt_row) else "RF"

    r2_sp_list, r2_full_list = [], []
    cv_attr = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    for train_idx, test_idx in cv_attr.split(X_arr_cv, stage_enc):
        # Species-only (no confounders, no imputation needed)
        m_sp   = sk_clone(models[best_mod])
        m_sp.fit(X_species_cv[train_idx], y[train_idx])
        r2_sp_list.append(r2_score(y[test_idx], m_sp.predict(X_species_cv[test_idx])))

        # Full model (species + confounders, fold-wise imputation)
        X_tr, X_te = _impute_fold(X_arr_cv[train_idx], X_arr_cv[test_idx], conf_col_indices)
        m_full = sk_clone(models[best_mod])
        m_full.fit(X_tr, y[train_idx])
        r2_full_list.append(r2_score(y[test_idx], m_full.predict(X_te)))

    r2_sp   = np.mean(r2_sp_list)
    r2_full = np.mean(r2_full_list)
    var_attr_rows.append({
        "target":          target,
        "target_name":     target_name,
        "r2_species_only": r2_sp,
        "r2_full":         r2_full,
        "r2_delta":        r2_full - r2_sp,
    })
    print(f"  {target_name[:35]:<35}  spe-only: {r2_sp:.3f}  full: {r2_full:.3f}  delta={r2_full-r2_sp:+.3f}")

var_attr = pd.DataFrame(var_attr_rows)
var_attr.to_csv(TABLE_DIR / "confounder_variance_attribution.csv", index=False)

# Stacked horizontal bar chart — do NOT clip negative R² (informative)
if not var_attr.empty:
    var_attr_plot = var_attr.copy().sort_values("r2_full", ascending=True)
    labels = var_attr_plot["target_name"].str[:35]

    # For the stacked bar, use max(0, r2) for bar lengths but annotate negatives
    r2_spe_bar  = var_attr_plot["r2_species_only"].clip(lower=0)
    r2_conf_bar = var_attr_plot["r2_delta"].clip(lower=0)

    fig, ax = plt.subplots(figsize=(9, max(5, len(var_attr_plot) * 0.45)))
    y_pos = range(len(var_attr_plot))
    ax.barh(list(y_pos), r2_spe_bar,
            color="#5C6BC0", label="Species (CLR)", height=0.55)
    ax.barh(list(y_pos), r2_conf_bar,
            left=r2_spe_bar,
            color="#FF7043", label="Confounders", height=0.55, alpha=0.85)

    # Annotate negative R² values
    for i, (_, row) in enumerate(var_attr_plot.iterrows()):
        if row["r2_species_only"] < 0:
            ax.text(-0.02, i, f'{row["r2_species_only"]:.2f}',
                    va="center", ha="right", fontsize=7, color="red")
        if row["r2_full"] < 0:
            ax.text(-0.02, i + 0.25, f'full: {row["r2_full"]:.2f}',
                    va="center", ha="right", fontsize=6, color="red")

    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel("Cross-validated R²")
    ax.set_title("Variance Attribution: Species vs Clinical Confounders", fontweight="bold")
    ax.legend(fontsize=9)
    ax.axvline(0, color="black", lw=0.6)
    plt.tight_layout()
    savefig(fig, "ml", "nb03_confounder_variance_attribution.png")

  C07151                               spe-only: -0.073  full: -0.073  delta=+0.001
  C00423                               spe-only: 0.092  full: 0.098  delta=+0.006
  C00122                               spe-only: 0.082  full: 0.086  delta=+0.005
  C01042                               spe-only: 0.246  full: 0.247  delta=+0.001
  C01959                               spe-only: -0.079  full: -0.081  delta=-0.002
  C08262                               spe-only: 0.026  full: 0.027  delta=+0.001
  C00024                               spe-only: 0.054  full: 0.069  delta=+0.015
  C00346                               spe-only: -0.134  full: -0.127  delta=+0.007
  C01879                               spe-only: 0.036  full: 0.030  delta=-0.006
  C16741                               spe-only: 0.034  full: 0.030  delta=-0.003
  C08277                               spe-only: 0.382  full: 0.383  delta=+0.002
  C01015                               spe-only: 0.082  full: 0.078  delta=-0.004
  C02037  

---
## 12 · Save ML Results

In [16]:
ml_results = {
    # Keys used by NB05 and NB06 — must match exactly
    "shap_summary":    shap_summary,    # wide DataFrame: metabolites x species
    "benchmark":       benchmark_df,    # long DataFrame: target x model metrics
    "targets":         all_targets,     # list of KEGG IDs (top dysregulated)
    "valid_mask":      valid_mask,      # boolean array for YACHIDA samples used in CV
    # Additional outputs
    "best_per_target": best_per_target,
    "shap_long":       shap_long,       # long-format SHAP table (for ad-hoc queries)
    "var_attr":        var_attr,
    "feature_names":   feature_names,
    "X_full":          X_full,
}
save_pickle(ml_results, INTER_DIR / "ml_results.pkl")

print("\n-- NB03 complete ---------------------------------------------------")
print(f"  Targets benchmarked      : {len(all_targets)}")
print(f"  Best model distribution  : {best_per_target['model'].value_counts().to_dict()}")
print(f"  SHAP results computed    : {len(shap_results)}")
print(f"  SHAP wide summary shape  : {shap_summary.shape}")
print(f"  Producer candidate rows  : {len(shap_long)}")
print("  -> Run NB04 (04_mofa_fda_trajectory.ipynb) next.")

Saved: C:\Users\andna\Documents\KI\Results\intermediate\ml_results.pkl

-- NB03 complete ---------------------------------------------------
  Targets benchmarked      : 50
  Best model distribution  : {'SVR': 27, 'RF': 16, 'LightGBM': 4, 'XGBoost': 2, 'ElasticNet': 1}
  SHAP results computed    : 49
  SHAP wide summary shape  : (49, 372)
  Producer candidate rows  : 960
  -> Run NB04 (04_mofa_fda_trajectory.ipynb) next.
